## 1. Import datasets 

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

#from sklift.datasets import fetch_x5

RAW_DIR = Path("../data/raw")

clients = pd.read_csv(RAW_DIR / "clients.csv")
uplift_train = pd.read_csv(RAW_DIR / "uplift_train.csv")

# Purchases are loaded later from the X5 dataset because the full transaction
# table is too large to keep in RAW_DIR.
# purchases = pd.read_csv(RAW_DIR / "purchases.csv")


In [2]:
print("clients:", clients.shape)
print("uplift_train:", uplift_train.shape)

clients: (400162, 5)
uplift_train: (200039, 3)


In [3]:
clients.head()

,client_id,first_issue_date,first_redeem_date,age,gender
0,000012768d,2017-08-05 15:40:48,2018-01-04 19:30:07,45,U
1,000036f903,2017-04-10 13:54:23,2017-04-23 12:37:56,72,F
2,000048b7a6,2018-12-15 13:33:11,NaN,68,F
3,000073194a,2017-05-23 12:56:14,2017-11-24 11:18:01,60,F
4,00007c7133,2017-05-22 16:17:08,2018-12-31 17:17:33,67,U


In [4]:
uplift_train.head()

,client_id,treatment_flg,target
0,000012768d,0,1
1,000036f903,1,1
2,00010925a5,1,1
3,0001f552b0,1,1
4,00020e7b18,1,1


### 1.1 Inspect the schemas

In [5]:
print("CLIENTS")
print(clients.dtypes)
print()

print("UPLIFT TRAIN")
print(uplift_train.dtypes)

CLIENTS
client_id              str
first_issue_date       str
first_redeem_date      str
age                  int64
gender                 str
dtype: object

UPLIFT TRAIN
client_id          str
treatment_flg    int64
target           int64
dtype: object


### 1.2 Verify the IDs

In [6]:
print("Unique client IDs in clients:",
      clients["client_id"].nunique())

print("Rows in clients:",
      len(clients))

print("Unique client IDs in uplift_train:",
      uplift_train["client_id"].nunique())

print("Rows in uplift_train:",
      len(uplift_train))

Unique client IDs in clients: 400162
Rows in clients: 400162
Unique client IDs in uplift_train: 200039
Rows in uplift_train: 200039


In [7]:
print("Missing client IDs:")
print(clients["client_id"].isna().sum())

print("Missing training client IDs:")
print(uplift_train["client_id"].isna().sum())

Missing client IDs:
0
Missing training client IDs:
0


In [8]:
missing_from_clients = uplift_train[
    ~uplift_train["client_id"].isin(clients["client_id"])
]

print("Training customers missing from clients:",
      len(missing_from_clients))

Training customers missing from clients: 0


In [9]:
clients_not_in_train = clients[
    ~clients["client_id"].isin(uplift_train["client_id"])
]

print("Clients not in training:",
      len(clients_not_in_train))

Clients not in training: 200123


Expected, as uplift_train.csv is a subset of the clients.csv

### 1.3 Construct the initial modelling table

In [10]:
modelling = uplift_train.merge(
    clients,
    on="client_id",
    how="left",
    validate="one_to_one",
)

In [11]:
print(modelling.shape)

(200039, 7)


In [12]:
modelling.head()

,client_id,treatment_flg,target,first_issue_date,first_redeem_date,age,gender
0,000012768d,0,1,2017-08-05 15:40:48,2018-01-04 19:30:07,45,U
1,000036f903,1,1,2017-04-10 13:54:23,2017-04-23 12:37:56,72,F
2,00010925a5,1,1,2018-07-24 16:21:29,2018-09-14 16:12:49,83,U
3,0001f552b0,1,1,2017-06-30 19:20:38,2018-08-28 12:59:45,33,F
4,00020e7b18,1,1,2017-11-27 11:41:45,2018-01-10 17:50:05,73,U


In [13]:
modelling.info()

<class 'pandas.DataFrame'>
RangeIndex: 200039 entries, 0 to 200038
Data columns (total 7 columns):
 #   Column             Non-Null Count   Dtype
---  ------             --------------   -----
 0   client_id          200039 non-null  str  
 1   treatment_flg      200039 non-null  int64
 2   target             200039 non-null  int64
 3   first_issue_date   200039 non-null  str  
 4   first_redeem_date  182493 non-null  str  
 5   age                200039 non-null  int64
 6   gender             200039 non-null  str  
dtypes: int64(3), str(4)
memory usage: 19.8 MB


### 1.4 Inspect Purchases 

In [14]:
from sklift.datasets import fetch_x5

dataset = fetch_x5()
purchases = dataset.data.purchases


In [15]:
print(purchases.shape)
print(purchases.dtypes)

(45786568, 13)
client_id                      str
transaction_id                 str
transaction_datetime           str
regular_points_received    float64
express_points_received    float64
regular_points_spent       float64
express_points_spent       float64
purchase_sum               float64
store_id                       str
product_id                     str
product_quantity           float64
trn_sum_from_iss           float64
trn_sum_from_red           float64
dtype: object


In [16]:
print(purchases.head())
print(
    purchases["transaction_datetime"].min(),
    purchases["transaction_datetime"].max()
)

    client_id transaction_id transaction_datetime  regular_points_received  \
0  000012768d     7e3e2e3984  2018-12-01 07:12:45                     10.0   
1  000012768d     7e3e2e3984  2018-12-01 07:12:45                     10.0   
2  000012768d     7e3e2e3984  2018-12-01 07:12:45                     10.0   
3  000012768d     7e3e2e3984  2018-12-01 07:12:45                     10.0   
4  000012768d     7e3e2e3984  2018-12-01 07:12:45                     10.0   

   express_points_received  regular_points_spent  express_points_spent  \
0                      0.0                   0.0                   0.0   
1                      0.0                   0.0                   0.0   
2                      0.0                   0.0                   0.0   
3                      0.0                   0.0                   0.0   
4                      0.0                   0.0                   0.0   

   purchase_sum    store_id  product_id  product_quantity  trn_sum_from_iss  \
0      

### 1.5 Construct the transaction-level table

In [17]:
purchases["transaction_datetime"] = pd.to_datetime(
    purchases["transaction_datetime"]
)

In [18]:
purchases_work = purchases.copy()

In [19]:
print("Rows:", len(purchases_work))
print("Unique customers:", purchases_work["client_id"].nunique())
print("Unique transactions:", purchases_work["transaction_id"].nunique())

print(
    "Date range:",
    purchases_work["transaction_datetime"].min(),
    "to",
    purchases_work["transaction_datetime"].max()
)

Rows: 45786568
Unique customers: 400162
Unique transactions: 8045201
Date range: 2018-11-21 21:02:33 to 2019-03-18 23:40:03


### 1.6 Check whether transaction IDs uniquely identify customers

In [20]:
transaction_customer_counts = (
    purchases_work
    .groupby("transaction_id")["client_id"]
    .nunique()
)

print(transaction_customer_counts.value_counts().sort_index())

client_id
1    8045173
2         28
Name: count, dtype: int64


In [21]:
ambiguous_transactions = (
    transaction_customer_counts[
        transaction_customer_counts > 1
    ].index
)

ambiguous_rows = (
    purchases_work[
        purchases_work["transaction_id"].isin(ambiguous_transactions)
    ]
    .sort_values(["transaction_id", "client_id"])
)

ambiguous_rows

,client_id,transaction_id,transaction_datetime,regular_points_received,express_points_received,regular_points_spent,express_points_spent,purchase_sum,store_id,product_id,product_quantity,trn_sum_from_iss,trn_sum_from_red
34279700,c031047e4b,03e2eae760,2018-11-25 09:33:15,0.7,0.0,0.0,0.0,151.00,1304408209,05fbeb7219,1.0,18.0,NaN
34279701,c031047e4b,03e2eae760,2018-11-25 09:33:15,0.7,0.0,0.0,0.0,151.00,1304408209,50851218e0,1.0,33.0,NaN
34279702,c031047e4b,03e2eae760,2018-11-25 09:33:15,0.7,0.0,0.0,0.0,151.00,1304408209,2a23f7e61c,1.0,18.0,NaN
34279703,c031047e4b,03e2eae760,2018-11-25 09:33:15,0.7,0.0,0.0,0.0,151.00,1304408209,27d35de863,1.0,42.0,NaN
34279704,c031047e4b,03e2eae760,2018-11-25 09:33:15,0.7,0.0,0.0,0.0,151.00,1304408209,882e01b160,0.0,41.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
11638684,416164263b,fdda0ad81b,2019-01-24 15:49:20,12.8,0.0,0.0,0.0,1289.65,7c6eadc2c5,1aec731556,0.0,186.0,NaN
21522723,78f51eafef,fdda0ad81b,2018-12-12 17:25:31,6.7,0.0,0.0,0.0,3526.00,f348d70716,78897d05f9,1.0,66.0,NaN
21522724,78f51eafef,fdda0ad81b,2018-12-12 17:25:31,6.7,0.0,0.0,0.0,3526.00,f348d70716,ef451a1d84,8.0,320.0,NaN
21522725,78f51eafef,fdda0ad81b,2018-12-12 17:25:31,6.7,0.0,0.0,0.0,3526.00,f348d70716,851e5a9155,1.0,220.0,NaN


The 28 cases have large time gaps, that gives strong evidence that transaction_id is simply a recycled identifier. </p>
A transaction should be identified by (`client_id`, `transaction_id`)

In [22]:
transaction_customer_counts = (
    purchases_work
    .groupby(["client_id", "transaction_id"])["client_id"]
    .nunique()
)

print(transaction_customer_counts.value_counts().sort_index())

client_id
1    8045229
Name: count, dtype: int64


## 2. Feature Engineering 

In [23]:
assert (transaction_customer_counts == 1).all()

transactions = (
    purchases_work
    .groupby((["client_id", "transaction_id"]), as_index=False)
    .agg(
        client_id=("client_id", "first"),
        transaction_datetime=("transaction_datetime", "first"),

        basket_value=("purchase_sum", "first"),
        n_items=("product_quantity", "sum"),

        regular_points_received=("regular_points_received", "first"),
        express_points_received=("express_points_received", "first"),
        regular_points_spent=("regular_points_spent", "first"),
        express_points_spent=("express_points_spent", "first"),

        store_id=("store_id", "first"),
        n_products_in_transaction=("product_id", "nunique"),
    )
)

In [24]:
# Check repeated basket values
basket_value_variation = (
    purchases_work
    .groupby(["client_id", "transaction_id"])["purchase_sum"]
    .nunique()
)

print(basket_value_variation.value_counts().sort_index())

purchase_sum
1    8045229
Name: count, dtype: int64


In [25]:
# Check repeated loyalty-point values
point_columns = [
    "regular_points_received",
    "express_points_received",
    "regular_points_spent",
    "express_points_spent",
]

for col in point_columns:
    variation = purchases_work.groupby(["client_id", "transaction_id"])[col].nunique()

    print(f"\n{col}")
    print(variation.value_counts().sort_index())

    assert (variation <= 1).all()


regular_points_received
regular_points_received
1    8045229
Name: count, dtype: int64

express_points_received
express_points_received
1    8045229
Name: count, dtype: int64

regular_points_spent
regular_points_spent
1    8045229
Name: count, dtype: int64

express_points_spent
express_points_spent
1    8045229
Name: count, dtype: int64


In [26]:
# Check transaction-level row count
assert len(transactions) == (
    purchases_work[["client_id", "transaction_id"]]
    .drop_duplicates()
    .shape[0]
)

assert not transactions.duplicated(
    ["client_id", "transaction_id"]
).any()

In [27]:
print(transactions.shape)
print(transactions.dtypes)
print(transactions.head())

(8045229, 11)
transaction_id                          str
client_id                               str
transaction_datetime         datetime64[us]
basket_value                        float64
n_items                             float64
regular_points_received             float64
express_points_received             float64
regular_points_spent                float64
express_points_spent                float64
store_id                                str
n_products_in_transaction             int64
dtype: object
  transaction_id   client_id transaction_datetime  basket_value  n_items  \
0     6a0e96d0bc  000012768d  2019-03-08 10:12:03         803.0     13.0   
1     7e3e2e3984  000012768d  2018-12-01 07:12:45        1007.0     21.0   
2     b34f23306e  000012768d  2019-03-14 15:01:47         419.0      6.0   
3     c1ca85d462  000012768d  2018-12-16 08:56:01         574.0     14.0   
4     0a3d640bf4  000036f903  2018-12-21 11:08:58         700.0      8.0   

   regular_points_received  exp

### 2.1 Construct transaction-level loyalty variables

In [28]:
transactions[point_columns].isna().sum()

regular_points_received    0
express_points_received    0
regular_points_spent       0
express_points_spent       0
dtype: int64

In [29]:
point_columns = [
    "regular_points_received",
    "express_points_received",
    "regular_points_spent",
    "express_points_spent",
]

transactions["points_earned"] = (
    transactions["regular_points_received"].fillna(0)
    + transactions["express_points_received"].fillna(0)
)

transactions["points_spent"] = abs(
    transactions["regular_points_spent"].fillna(0)
    + transactions["express_points_spent"].fillna(0)
)

transactions["points_earned_flag"] = (
    transactions["points_earned"] > 0
)

transactions["points_spent_flag"] = (
    transactions["points_spent"] > 0
)

### 2.2 Customer-level RFM and breadth

- Number of transactions = number of unique `transaction_id` per `client_id`
- Total spending = sum of `purchase_sum` per `client_id`
- Average basket value = mean of total `purchase_sum` per `transaction_id`
- Median basket value = median of total `purchase_sum` per `transaction_id`
- Average number of items/basket = mean of total `product_quantity` per `transaction_id`
- Median number of items/basket = median of total `product_quantity` per `transaction_id`
- Stores visited = nunique(`store_id`)
- Points earned = regular points received + express points received
- Points spent = regular points spent + express points spent
- Proportion of purchases with points spent = #transactions / #transactions where points spent > 0
- Proportion of purchases with points spent = #transactions / #transactions where points spent > 0

In [30]:
customer_features = (
    transactions
    .groupby("client_id")
    .agg(
        # RFM-style behavior
        n_transactions=("transaction_id", "size"),
        total_spending=("basket_value", "sum"),
        avg_basket_value=("basket_value", "mean"),
        median_basket_value=("basket_value", "median"),
        avg_items_per_basket=("n_items", "mean"),
        median_items_per_basket=("n_items", "median"),

        # Breadth
        n_stores=("store_id", "nunique"),

        # Loyalty
        loyalty_points_earned=("points_earned", "sum"),
        loyalty_points_spent=("points_spent", "sum"),
        prop_transactions_points_earned=("points_earned_flag", "mean"),
        prop_transactions_points_spent=("points_spent_flag", "mean"),

        # Dates
        first_purchase_date=("transaction_datetime", "min"),
        last_purchase_date=("transaction_datetime", "max"),
    )
    .reset_index()
)

### 2.3 Temporal behavior

- observation period in months = (time last transaction - time first transaction) / 30.44
- purchase frequency common per month = # transactions / overall dataset observation period in months
- 

In [31]:
distinct_products = (
    purchases_work
    .groupby("client_id")["product_id"]
    .nunique()
    .rename("n_distinct_products")
    .reset_index()
)

customer_features = customer_features.merge(
    distinct_products,
    on="client_id",
    how="left",
    validate="one_to_one",
)

In [32]:
# Observation period 
customer_features["observation_period_days"] = (
    customer_features["last_purchase_date"]
    - customer_features["first_purchase_date"]
).dt.total_seconds() / (24 * 60 * 60)

customer_features["observation_period_months"] = (
    customer_features["observation_period_days"] / 30.44
)

In [33]:
# Common observation period
common_start = transactions["transaction_datetime"].min()
common_end = transactions["transaction_datetime"].max()

common_observation_months = (
    (common_end - common_start).total_seconds()
    / (24 * 60 * 60)
    / 30.44
)

print(
    "Common observation period:",
    common_start,
    "to",
    common_end
)

print(
    "Common observation period (months):",
    common_observation_months
)

Common observation period: 2018-11-21 21:02:33 to 2019-03-18 23:40:03
Common observation period (months): 3.8472199408672796


In [34]:
# 30/60/90 Features 
reference_date = pd.Timestamp("2019-03-19 00:00:00")

for days in [30, 60, 90]:

    start_date = reference_date - pd.Timedelta(days=days)

    mask = (
        (transactions["transaction_datetime"] >= start_date)
        & (transactions["transaction_datetime"] < reference_date)
    )

    window = transactions.loc[mask]

    counts = (
        window
        .groupby("client_id")
        .size()
        .rename(f"transactions_last_{days}d")
    )

    spending = (
        window
        .groupby("client_id")["basket_value"]
        .sum()
        .rename(f"spending_last_{days}d")
    )

    customer_features = customer_features.merge(
        counts,
        on="client_id",
        how="left",
        validate="one_to_one",
    )

    customer_features = customer_features.merge(
        spending,
        on="client_id",
        how="left",
        validate="one_to_one",
    )

In [35]:
temporal_columns = [
    col
    for col in customer_features.columns
    if col.startswith("transactions_last_")
    or col.startswith("spending_last_")
]

customer_features[temporal_columns] = (
    customer_features[temporal_columns].fillna(0)
)

In [36]:
# Inter-purchase intervals
transactions_sorted = transactions.sort_values(
    ["client_id", "transaction_datetime"]
).copy()

transactions_sorted["days_since_previous_purchase"] = (
    transactions_sorted
    .groupby("client_id")["transaction_datetime"]
    .diff()
    .dt.total_seconds()
    / (24 * 60 * 60)
)

In [37]:
purchase_intervals = (
    transactions_sorted
    .groupby("client_id")["days_since_previous_purchase"]
    .agg(
        mean_days_between_purchases="mean",
        median_days_between_purchases="median",
        min_days_between_purchases="min",
        max_days_between_purchases="max",
        std_days_between_purchases="std",
    )
    .reset_index()
)

customer_features = customer_features.merge(
    purchase_intervals,
    on="client_id",
    how="left",
    validate="one_to_one",
)

## 3. Final Feature Table

In [38]:
print(customer_features.shape)
print(customer_features.columns.tolist())
print(customer_features.head())

(400162, 28)
['client_id', 'n_transactions', 'total_spending', 'avg_basket_value', 'median_basket_value', 'avg_items_per_basket', 'median_items_per_basket', 'n_stores', 'loyalty_points_earned', 'loyalty_points_spent', 'prop_transactions_points_earned', 'prop_transactions_points_spent', 'first_purchase_date', 'last_purchase_date', 'n_distinct_products', 'observation_period_days', 'observation_period_months', 'transactions_last_30d', 'spending_last_30d', 'transactions_last_60d', 'spending_last_60d', 'transactions_last_90d', 'spending_last_90d', 'mean_days_between_purchases', 'median_days_between_purchases', 'min_days_between_purchases', 'max_days_between_purchases', 'std_days_between_purchases']
    client_id  n_transactions  total_spending  avg_basket_value  \
0  000012768d               4         2803.00        700.750000   
1  000036f903              32         9805.00        306.406250   
2  000048b7a6               8         3772.00        471.500000   
3  000073194a              17

### 3.1 Validate this feature table

In [ ]:
# Missing values & Summary statistics
print(customer_features.isna().sum().sort_values(ascending=False))

print(
    customer_features[
        [
            "n_transactions",
            "total_spending",
            "avg_basket_value",
            "avg_items_per_basket",
            "observation_period_months",
            "n_distinct_products",
            "n_stores",
            "loyalty_points_earned",
            "loyalty_points_spent",
        ]
    ].describe().T
)


std_days_between_purchases         18785
max_days_between_purchases          7707
min_days_between_purchases          7707
median_days_between_purchases       7707
mean_days_between_purchases         7707
n_transactions                         0
spending_last_90d                      0
transactions_last_90d                  0
spending_last_60d                      0
transactions_last_60d                  0
spending_last_30d                      0
transactions_last_30d                  0
observation_period_months              0
observation_period_days                0
client_id                              0
last_purchase_date                     0
first_purchase_date                    0
prop_transactions_points_spent         0
prop_transactions_points_earned        0
loyalty_points_spent                   0
loyalty_points_earned                  0
n_stores                               0
median_items_per_basket                0
avg_items_per_basket                   0
median_basket_va

In [41]:
print(
    "Customers with zero observation period:",
    (customer_features["observation_period_days"] == 0).sum()
)

print(
    "Customers with one transaction:",
    (customer_features["n_transactions"] == 1).sum()
)

print(
    "Customers with two transactions:",
    (customer_features["n_transactions"] == 2).sum()
)

print(
    "Customers with missing interval std:",
    customer_features["std_days_between_purchases"].isna().sum()
)

Customers with zero observation period: 7707
Customers with one transaction: 7707
Customers with two transactions: 11078
Customers with missing interval std: 18785


Standard deviation with pandas' default ddof=1 requires at least two observations.
- there are 7,707 customers with one transaction. Those customers have no interval at all, so their standard deviation is naturally undefined
- additional missing values are the customers with exactly two transactions. Variability can't be estimated from a single interval, so pandas returns NaN

The same principle applies to the other interval features:
- mean/median/min/max → defined with ≥2 transactions
- standard deviation → defined with ≥3 transactions

No imputation needed!

In [42]:
customer_features[
    [
        "n_transactions",
        "observation_period_days",
        "observation_period_months",
        "first_purchase_date",
        "last_purchase_date",
    ]
].sort_values(
    "observation_period_days"
).head(20)


,n_transactions,observation_period_days,observation_period_months,first_purchase_date,last_purchase_date
152437,1,0.0,0.0,2019-03-15 17:00:35,2019-03-15 17:00:35
118247,1,0.0,0.0,2019-03-08 17:59:50,2019-03-08 17:59:50
19014,1,0.0,0.0,2019-03-10 15:48:56,2019-03-10 15:48:56
345691,1,0.0,0.0,2019-03-09 08:17:24,2019-03-09 08:17:24
235806,1,0.0,0.0,2019-03-11 13:13:34,2019-03-11 13:13:34
217329,1,0.0,0.0,2019-03-11 09:28:24,2019-03-11 09:28:24
59662,1,0.0,0.0,2019-03-14 09:20:38,2019-03-14 09:20:38
314751,1,0.0,0.0,2019-03-06 11:16:16,2019-03-06 11:16:16
118288,1,0.0,0.0,2019-03-03 07:36:49,2019-03-03 07:36:49
383134,1,0.0,0.0,2019-03-08 12:09:32,2019-03-08 12:09:32


In [43]:
print(
    customer_features["observation_period_days"]
    .describe(
        percentiles=[
            0.001,
            0.005,
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
        ]
    )
)

count    400162.000000
mean         91.345400
std          28.492124
min           0.000000
0.1%          0.000000
0.5%          0.000000
1%            0.000000
5%           20.349213
10%          46.024065
25%          84.211658
50%         102.230116
max         116.870197
Name: observation_period_days, dtype: float64


In [44]:
transactions[
    [
        "regular_points_spent",
        "express_points_spent",
        "points_spent",
    ]
].describe()

,regular_points_spent,express_points_spent,points_spent
count,8.045229e+06,8.045229e+06,8.045229e+06
mean,-3.651405e+00,-3.192782e-01,3.970683e+00
std,2.557845e+01,3.245818e+00,2.615275e+01
min,-5.066000e+03,-3.000000e+02,0.000000e+00
25%,0.000000e+00,0.000000e+00,0.000000e+00
50%,0.000000e+00,0.000000e+00,0.000000e+00
75%,0.000000e+00,0.000000e+00,0.000000e+00
max,0.000000e+00,0.000000e+00,5.066000e+03


In [45]:
print(
    "Positive regular points spent:",
    (transactions["regular_points_spent"] > 0).sum()
)

print(
    "Positive express points spent:",
    (transactions["express_points_spent"] > 0).sum()
)

print(
    "Negative regular points spent:",
    (transactions["regular_points_spent"] < 0).sum()
)

print(
    "Negative express points spent:",
    (transactions["express_points_spent"] < 0).sum()
)

print(
    "Transactions with points spent:",
    (transactions["points_spent"] > 0).sum()
)

print(
    "Transactions without points spent:",
    (transactions["points_spent"] == 0).sum()
)

print(
    "Total points spent:",
    transactions["points_spent"].sum()
)

Positive regular points spent: 0
Positive express points spent: 0
Negative regular points spent: 495331
Negative express points spent: 90469
Transactions with points spent: 504597
Transactions without points spent: 7540632
Total points spent: 31945054.0


## 4. Merge with Train 

In [46]:
print(customer_features.shape)
print(customer_features["client_id"].is_unique)
print(customer_features["client_id"].nunique())

(400162, 28)
True
400162


In [47]:
assert customer_features["client_id"].is_unique

In [48]:
missing_features = (
    uplift_train["client_id"]
    .isin(customer_features["client_id"])
    .sum()
)

print("Training customers without features:", len(uplift_train) - missing_features)

Training customers without features: 0


In [49]:
assert uplift_train["client_id"].isin(
    customer_features["client_id"]
).all()

In [50]:
modelling = uplift_train.merge(
    customer_features,
    on="client_id",
    how="left",
    validate="one_to_one",
)

assert len(modelling) == len(uplift_train)
assert modelling["client_id"].is_unique

In [51]:
modelling = modelling.merge(
    clients,
    on="client_id",
    how="left",
    validate="one_to_one",
)

assert len(modelling) == len(uplift_train)
assert modelling["client_id"].is_unique

In [52]:
modelling["first_issue_date"] = pd.to_datetime(
    modelling["first_issue_date"]
)

modelling["first_redeem_date"] = pd.to_datetime(
    modelling["first_redeem_date"]
)

modelling["issue_redeem_delay"] = (
    modelling["first_redeem_date"] - modelling["first_issue_date"]
).dt.total_seconds() / (24 * 60 * 60)

In [53]:
print("MODEllING")
print(modelling.info())

MODEllING
<class 'pandas.DataFrame'>
RangeIndex: 200039 entries, 0 to 200038
Data columns (total 35 columns):
 #   Column                           Non-Null Count   Dtype         
---  ------                           --------------   -----         
 0   client_id                        200039 non-null  str           
 1   treatment_flg                    200039 non-null  int64         
 2   target                           200039 non-null  int64         
 3   n_transactions                   200039 non-null  int64         
 4   total_spending                   200039 non-null  float64       
 5   avg_basket_value                 200039 non-null  float64       
 6   median_basket_value              200039 non-null  float64       
 7   avg_items_per_basket             200039 non-null  float64       
 8   median_items_per_basket          200039 non-null  float64       
 9   n_stores                         200039 non-null  int64         
 10  loyalty_points_earned            200039 non-n

In [54]:
# Export dataset
output_path = Path("../data/interim/modelling_dataset.parquet")

output_path.parent.mkdir(parents=True, exist_ok=True)

modelling.to_parquet(output_path, engine="pyarrow", index=False)

print(f"Saved to: {output_path}")
print(f"Shape: {modelling.shape}")

Saved to: ../data/interim/modelling_dataset.parquet
Shape: (200039, 35)
